# 8장 상태전이(마르코프) 모형 (R과 통계분석)

보건통계학 기초 23장 b 절의 마르코프 모형(무진행, 진행, 사망)을 R의 행렬 계산으로 만들고, 7장의 분할생존모형과 비교합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요.


## a. 준비

### 셀 1. 입력값과 결과 계산 함수 (7장과 같음)

In [ ]:
weib_lam <- function(median, gam) log(2) / median^gam
p <- list(pfs_gam = 0.95, pfs_lam = weib_lam(10, 0.95), os_gam = 1.15, os_lam = weib_lam(28, 1.15),
          hr_pfs = 0.65, hr_os = 0.75, u_pf = 0.78, u_pd = 0.62, du_ae_A = 0.012, du_ae_B = 0.008,
          c_drug_A = 190, c_drug_B = 120, c_pf = 40, c_pd = 250, c_death = 800,
          c_ae_A = 120, c_ae_B = 80, disc = 0.045)            # 7장과 같은 입력값
mid <- function(x) (head(x, -1) + tail(x, -1)) / 2
arm_result <- function(p, a, pf_t, pd_t, half = TRUE) {   # 상태 비율(0-n개월) -> 한 군의 결과
  n <- length(pf_t) - 1
  pf <- if (half) mid(pf_t) else head(pf_t, -1)
  pd <- if (half) mid(pd_t) else head(pd_t, -1)
  alive <- pf_t + pd_t
  died <- head(alive, -1) - tail(alive, -1)
  disc <- 1 / (1 + p$disc)^((0:(n - 1) + 0.5) / 12)
  cost <- sum((pf * (p[[paste0("c_drug_", a)]] + p$c_pf) + pd * p$c_pd +
               died * p$c_death) * disc) + p[[paste0("c_ae_", a)]]
  qaly <- sum((pf * p$u_pf + pd * p$u_pd) / 12 * disc) - p[[paste0("du_ae_", a)]]
  c(ly_pf = sum(pf) / 12, ly_pd = sum(pd) / 12, ly = sum(pf + pd) / 12, qaly = qaly, cost = cost)
}
compare <- function(rA, rB) {
  tab <- cbind(A = rA, B = rB, diff = rA - rB)
  list(table = tab, icer = unname(tab["cost", "diff"] / tab["qaly", "diff"]))
}
t <- 0:240
S <- function(lam, gam, t) exp(-lam * t^gam)
psm_arm <- function(a) {                              # 7장의 분할생존모형
  h1 <- if (a == "A") p$hr_pfs else 1; h2 <- if (a == "A") p$hr_os else 1
  os <- S(p$os_lam * h2, p$os_gam, t); pf <- pmin(S(p$pfs_lam * h1, p$pfs_gam, t), os)
  arm_result(p, a, pf, os - pf)
}
psm <- compare(psm_arm("A"), psm_arm("B"))
round(psm$icer, 2)

## b. 율과 확률

### 셀 2. 율과 확률 바꾸기 (23장 표 23-7)

In [ ]:
rate_to_prob <- function(r, t = 1) 1 - exp(-r * t)        # 율 -> 길이 t 기간의 확률
prob_to_rate <- function(pr, t = 1) -log(1 - pr) / t       # 확률 -> 율
r_exit <- log(2) / 10                                      # 중앙 무진행생존 10개월 -> 월 율
p_year  <- rate_to_prob(r_exit, 12)
p_month <- rate_to_prob(r_exit)
round(c(rate = r_exit, p_year = p_year, p_month = p_month), 4)
round(rbind(month_from_year = c(right = rate_to_prob(prob_to_rate(p_year, 12)), wrong = p_year / 12),
            year_from_month = c(right = 1 - (1 - p_month)^12, wrong = p_month * 12),
            apply_hr        = c(right = rate_to_prob(r_exit * p$hr_pfs, 12), wrong = p_year * p$hr_pfs)), 4)

## c. 전이확률 행렬과 코호트 추적

### 셀 3. 율에서 전이확률 행렬 만들기

In [ ]:
tmat <- function(r_prog, r_dpf, r_dpd) {       # 월 율 세 개 -> 한 달 전이확률 행렬
  leave <- rate_to_prob(r_prog + r_dpf)          # 무진행 상태를 떠날 확률
  to_pd <- leave * r_prog / (r_prog + r_dpf)     # 그 가운데 진행으로 가는 몫
  q <- rate_to_prob(r_dpd)                       # 진행 상태에서 사망할 확률
  matrix(c(1 - leave, to_pd, leave - to_pd,
           0,         1 - q, q,
           0,         0,     1), nrow = 3, byrow = TRUE,
         dimnames = list(from = c("pf", "pd", "dead"), to = c("pf", "pd", "dead")))
}
r_dpd <- 0.0470602                               # 진행 -> 사망의 월 율 (23장)
P_B <- tmat(0.9 * r_exit, 0.1 * r_exit, r_dpd)
P_A <- tmat(0.9 * r_exit * p$hr_pfs, 0.1 * r_exit * p$hr_pfs, r_dpd)   # 위험비는 율에 곱한다
round(P_B, 4)
round(P_A, 4)
rbind(B = rowSums(P_B), A = rowSums(P_A))

### 셀 4. 코호트 추적 (표준요법 B)

In [ ]:
trace <- function(P, n = 240) {                  # 모두 무진행에서 출발해 n주기 추적
  x <- matrix(0, n + 1, 3, dimnames = list(NULL, colnames(P)))
  x[1, ] <- c(1, 0, 0)
  for (k in 1:n) x[k + 1, ] <- x[k, ] %*% P      # 행렬 곱 한 번 = 한 달
  x
}
tr_B <- trace(P_B); tr_A <- trace(P_A)
dim(tr_B)
round(tr_B[c(0, 1, 2, 3, 12, 24) + 1, ], 3)
P24 <- diag(3)
for (k in 1:24) P24 <- P24 %*% P_B               # 행렬을 24번 곱한 것
round(rbind(power = P24[1, ], trace = tr_B[24 + 1, ]), 4)

### 셀 5. 모형 속 중앙 전체생존

In [ ]:
median_os <- function(P, n = 1200) {            # 생존 비율이 0.5가 되는 시점(개월)
  alive <- rowSums(trace(P, n)[, c("pf", "pd")])
  k <- which(alive <= 0.5)[1]                     # 처음으로 0.5 이하가 된 행 (k행 = k - 1개월)
  (k - 2) + (alive[k - 1] - 0.5) / (alive[k - 1] - alive[k])
}
round(c(B = median_os(P_B), A = median_os(P_A)), 1)

## d. 비용, QALY, ICER

### 셀 6. 마르코프 모형의 결과

In [ ]:
mk <- compare(arm_result(p, "A", tr_A[, "pf"], tr_A[, "pd"]),
              arm_result(p, "B", tr_B[, "pf"], tr_B[, "pd"]))
round(mk$table, 3)
round(mk$icer)
nohalf <- compare(arm_result(p, "A", tr_A[, "pf"], tr_A[, "pd"], half = FALSE),
                  arm_result(p, "B", tr_B[, "pf"], tr_B[, "pd"], half = FALSE))
round(c(ly_B = unname(nohalf$table["ly", "B"]), icer = nohalf$icer), 3)

### 셀 7. 두 모형의 전체생존 곡선

In [ ]:
os_psm <- cbind(B = S(p$os_lam, p$os_gam, t), A = S(p$os_lam * p$hr_os, p$os_gam, t))
os_mk  <- cbind(B = rowSums(tr_B[, 1:2]), A = rowSums(tr_A[, 1:2]))
matplot(t / 12, cbind(os_psm, os_mk), type = "l", lty = c(1, 1, 2, 2),
        col = c("#c8553d", "#1f5f8b"), xlim = c(0, 15), ylim = c(0, 1),
        xlab = "Years", ylab = "Overall survival")
legend("topright", c("B, Weibull (partitioned)", "A, Weibull (partitioned)",
                     "B, Markov", "A, Markov"),
       col = c("#c8553d", "#1f5f8b"), lty = c(1, 1, 2, 2), bty = "n")

## e. 맞추기와 두 구조의 비교

### 셀 8. 진행 후 사망을 군별로 맞추기와 두 구조의 비교

In [ ]:
gap <- function(r, hr, target)                   # 진행 -> 사망 율이 r일 때 중앙 OS - 목표
  median_os(tmat(0.9 * r_exit * hr, 0.1 * r_exit * hr, r)) - target
med_B <- (log(2) / p$os_lam)^(1 / p$os_gam)
med_A <- (log(2) / (p$os_lam * p$hr_os))^(1 / p$os_gam)
r_B <- uniroot(gap, c(0.01, 1), hr = 1, target = med_B, tol = 1e-10)$root
r_A <- uniroot(gap, c(0.01, 1), hr = p$hr_pfs, target = med_A, tol = 1e-10)$root
round(c(med_B = med_B, r_B = r_B, med_A = med_A, r_A = r_A), 5)
tr_A2 <- trace(tmat(0.9 * r_exit * p$hr_pfs, 0.1 * r_exit * p$hr_pfs, r_A))
mk2 <- compare(arm_result(p, "A", tr_A2[, "pf"], tr_A2[, "pd"]), mk$table[, "B"])
row <- function(r) c(d_pf = r$table["ly_pf", "diff"], d_pd = r$table["ly_pd", "diff"],
                     d_cost = r$table["cost", "diff"], d_qaly = r$table["qaly", "diff"], icer = r$icer)
cmp <- rbind(Markov = row(mk), Markov_by_arm = row(mk2), Partitioned = row(psm))
cmp <- round(cmp, 3); cmp[, "icer"] <- round(cmp[, "icer"]); cmp

## f. 과제 정답

### 셀 9. 과제 1 정답: 1년 확률을 12로 나누면

In [ ]:
p_wrong <- p_year / 12                           # 동료가 넣은 한 달 확률
r_w <- prob_to_rate(p_wrong)
tB <- trace(tmat(0.9 * r_w, 0.1 * r_w, r_dpd))
tA <- trace(tmat(0.9 * r_w * p$hr_pfs, 0.1 * r_w * p$hr_pfs, r_dpd))
wrong <- compare(arm_result(p, "A", tA[, "pf"], tA[, "pd"]), arm_result(p, "B", tB[, "pf"], tB[, "pd"]))
round(c(p_month_wrong = p_wrong, p_month_right = p_month, median_pfs_B = log(2) / r_w,
        median_os_B = median_os(tmat(0.9 * r_w, 0.1 * r_w, r_dpd))), 4)
round(c(ly_B = unname(wrong$table["ly", "B"]), d_cost = unname(wrong$table["cost", "diff"]),
        d_qaly = unname(wrong$table["qaly", "diff"]), icer = wrong$icer), 3)

### 셀 10. 과제 2 정답: 시간에 따라 변하는 전이확률

In [ ]:
trace_tv <- function(hr, n = 240) {             # 무진행을 떠날 확률이 주기마다 다른 모형
  s <- S(p$pfs_lam * hr, p$pfs_gam, 0:n)          # 와이블 PFS
  q <- rate_to_prob(r_dpd)
  x <- matrix(0, n + 1, 3, dimnames = list(NULL, c("pf", "pd", "dead")))
  x[1, ] <- c(1, 0, 0)
  for (k in 1:n) {
    leave <- 1 - s[k + 1] / s[k]                  # k번째 주기에 무진행을 떠날 확률
    P <- matrix(c(1 - leave, 0.9 * leave, 0.1 * leave,
                  0, 1 - q, q,
                  0, 0, 1), nrow = 3, byrow = TRUE)
    x[k + 1, ] <- x[k, ] %*% P
  }
  x
}
tvB <- trace_tv(1); tvA <- trace_tv(p$hr_pfs)
tv <- compare(arm_result(p, "A", tvA[, "pf"], tvA[, "pd"]), arm_result(p, "B", tvB[, "pf"], tvB[, "pd"]))
round(tv$table[c("ly_pf", "ly_pd", "ly", "qaly", "cost"), ], 3)
round(tv$icer)

### 셀 11. 과제 3 정답: 이상반응 예방 요법의 결정수형

In [ ]:
ae_tree <- function(p_ae = 0.30, rr = 0.60, p_hosp = 0.40, c_hosp = 700, c_out = 200,
                    dq_hosp = 0.07, dq_out = 0.02, c_proph = 60) {
  one <- function(p_e, add) {
    prob <- c(hosp = p_e * p_hosp, out = p_e * (1 - p_hosp), none = 1 - p_e)
    c(cost = sum(prob * (add + c(c_hosp, c_out, 0))), qaly = -sum(prob * c(dq_hosp, dq_out, 0)))
  }
  r <- rbind(none = one(p_ae, 0), proph = one(p_ae * rr, c_proph))
  d <- r["proph", ] - r["none", ]
  list(table = r, icer = unname(d["cost"] / d["qaly"]), inmb_5000 = unname(5000 * d["qaly"] - d["cost"]))
}
ae_tree()